# Example 1-CH: Cahn Hilliard patterns for 2D aggregation-diffusion

In this case, we consider a simple 2D geometry comprised of two compartments:
- surf - 2D surface
- edge - outer edges of the surface (1D)

We implement a Cahn-Hilliard model with two species, one exhibiting aggregation-diffusion ($B$) and the other purely diffusive ($X$).
The equations governing their evolution are given by:

$$
\partial_t{u_X} = -k_{on} u_X + k_{off} u_B + D_X \nabla \cdot (\hat{\mu}_X \nabla u_X) \\
\partial_t{u_B} = k_{on} u_X - k_{off} u_B + D_B \nabla \cdot (\hat{\mu}_B \nabla u_B),
$$

where aggregation is accounted for through the chemical potential of $B$, whose nondimensional version ($\hat{\mu}_B = \frac{\mu_B}{k_B T}$) is given by

$$
\hat{\mu}_B = (\ln \phi_B - \ln (1-\phi_B)) - \hat{A} (2 \phi_B - 1) - \frac{\hat{A}}{u_{B,max}} \nabla^2 \phi_B
$$

in which $\phi_B$ is the area fraction occupied by species B, which is proportional to its concentration; that is, $\phi_B = \frac{u_B}{u_{B,max}}$.
The chemical potential is defined similarly for X, but with $\hat{A}=0$; that is, $\hat{\mu}_X = (\ln \phi_X - \ln (1-\phi_X))$

We solve these equations over a square domain with no-flux boundary conditions.

We begin with the necessary imports:

In [ ]:
import dolfin as d
import sympy as sym
import numpy as np
import pathlib
import gmsh  # must be imported before pyvista if dolfin is imported first

from smart import config, common, mesh, model, mesh_tools, visualization
from smart.units import unit
from smart.model_assembly import (
    Compartment,
    Parameter,
    Reaction,
    Species,
    SpeciesContainer,
    ParameterContainer,
    CompartmentContainer,
    ReactionContainer,
)
import logging
from matplotlib import pyplot as plt

We will set the logging level to `INFO`. This will display some output during the simulation. If you want to get even more output you could set the logging level to `DEBUG`.

In [ ]:
logger = logging.getLogger("smart")
logger.setLevel(logging.INFO)

Futhermore, you could also save the logs to a file by attaching a file handler to the logger as follows.

```
file_handler = logging.FileHandler("filename.log")
file_handler.setFormatter(logging.Formatter(smart.config.base_format))
logger.addHandler(file_handler)
```

We define the various units for use in the model. 

In [ ]:
# Aliases - base units
um = unit.um
molecule = unit.molecule
sec = unit.sec
dimensionless = unit.dimensionless
D_unit = um**2 / sec
flux_unit = molecule / (um * sec)
surf_unit = molecule / um**2
edge_unit = molecule / um

## Generate model

### Compartments
As described above, the two compartments are the "surf" (2D) and edge (1D). These are initialized by calling:
```
compartment_var = Compartment(name, dimensionality, compartment_units, cell_marker)
```
where
- name: string naming the compartment
- dimensionality: topological dimensionality (e.g. 2 for surf, 1 for edge)
- compartment_units: length units for the compartment (um for both here)
- cell_marker: integer marker value identifying each compartment in the parent mesh

In [ ]:
surf = Compartment("surf", 2, um, 10)

Now we initialize a compartment container and add both compartments to it.

In [ ]:
cc = CompartmentContainer()
cc.add([surf])

### Species
In this case, we have a two species, "X" and "B", which exist in the 2D "surf" domain. Each is initialized by calling:
```
species_var = Species(
            name, initial_condition, concentration_units,
            D, diffusion_units, compartment_name, group (opt)
        )
```
where
- name: string naming the species
- initial_condition: initial concentration for this species (can be an expression given by a string to be parsed by sympy - the only unknowns in the expression should be x, y, and z)
- concentration_units: concentration units for this species (molecules/μm<sup>2</sup> here)
- D: diffusion coefficient
- diffusion_units: units for diffusion coefficient (μm<sup>2</sup>/sec here)
- compartment_name: each species should be assigned to a single compartment ("surf", here)
- group (opt): for larger models, specifies a group of species this belongs to;
            for organizational purposes when there are multiple reaction modules

With the added CH features, we also must provide the following for Cahn-Hilliard type species:
- `CH = True` - this tells SMART that we are considering both aggregation and diffusion
- umax: maximum surface density of X or B
- A_hat: strength of aggregation

Note that A_hat is dimensionless here and the chemical potential is a variable generated *internally* within this branch of SMART. Because only the nondimensional chemical potential appears in the dynamical equation for $u_A$, the chemical potential is always normalized to the thermal energy scale $k_B T$. (see equations up top for consistency on this point)

In [ ]:
l0 = np.sqrt(4*np.pi) # reference length scale
Shat = 10.0
phi0_X = 0.1
phi0_B = 0.1
sigma_s = Shat/l0**2
X = Species("X", phi0_X*sigma_s, surf_unit, 1.0, D_unit, "surf", CH=True, umax=sigma_s, A_hat = 0) 
B = Species("B", phi0_B*sigma_s, surf_unit, 1.0, D_unit, "surf", CH=True, umax=sigma_s, A_hat = 50)

Create a species container and add both species to it:

In [ ]:
sc = SpeciesContainer()
sc.add([X, B])

### Parameters and Reactions
Parameters and reactions are generally defined together, although the order does not strictly matter. Parameters are specified as:
```
param_var = Parameter(name, value, unit, group (opt), notes (opt), use_preintegration (opt))
```
where
- name: string naming the parameter
- value: value of the given parameter
- unit: units associated with given value
- group (optional): optional string placing this reaction in a reaction group; for organizational purposes when there are multiple reaction modules
- notes (optional): string related to this parameter
- use_preintegration (optional): in the case of a time-dependent parameter, uses preintegration in the solution process

Reactions are specified by a variable number of arguments (arguments are indicated by (opt) are either never
required or only required in some cases, for more details see notes below and API documentation):
```
reaction_var = Reaction(
                name, lhs, rhs, param_map,
                eqn_f_str (opt), eqn_r_str (opt), reaction_type (opt), species_map,
                explicit_restriction_to_domain (opt), group (opt), flux_scaling (opt)
            )
```
- name: string naming the reaction
- lhs: list of strings specifying the reactants for this reaction
- rhs: list of strings specifying the products for this reaction
    ***NOTE: the lists "reactants" and "products" determine the stoichiometry of the reaction;
       for instance, if two A's react to give one B, the reactants list would be ["A","A"],
       and the products list would be ["B"]
- param_map: relationship between the parameters specified in the reaction string and those given
              in the parameter container. By default, the reaction parameters are "kon" and "koff" when
              a system obeys simple mass action. If the forward rate is given by a parameter "k1" and the
              reverse rate is given by "k2", then param_map = {"on":"k1", "off":"k2"}
- eqn_f_str: For systems not obeying simple mass action, this string specifies the forward reaction rate
             By default, this string is "on*{all reactants multiplied together}"
- eqn_r_str: For systems not obeying simple mass action, this string specifies the reverse reaction rate
             By default, this string is "off*{all products multiplied together}"
- reaction_type (opt): either "custom" or "mass_action" (default is "mass_action") [never a required argument]
- species_map: same format as param_map; required if other species not listed in reactants or products appear in the
            reaction string
- explicit_restriction_to_domain: string specifying where the reaction occurs; required if the reaction is not
                                  constrained by the reaction string (e.g., if production occurs only at the boundary,
                                  as it does here, but the species being produced exists through the entire volume)
- group (opt): string placing this reaction in a reaction group; for organizational purposes when there are multiple reaction modules
- flux_scaling (opt): in certain cases, a given reactant or product may experience a scaled flux (for instance, if we assume that
                some of the molecules are immediately sequestered after the reaction); in this case, to signify that this flux 
                should be rescaled, we specify ''flux_scaling = {scaled_species: scale_factor}'', where scaled_species is a
                string specifying the species to be scaled and scale_factor is a number specifying the rescaling factor

For this system, we do not define any reactions on the boundary (`edge`). This corresponds to assuming a no-flux boundary condition.

In [ ]:
kon_hat = 1.0
koff_hat = 1.0
tref = l0**2 / float(X.D)
kon = Parameter("kon", kon_hat*sigma_s**(3/2)/tref, 1/sec)
koff = Parameter("koff", koff_hat*sigma_s**(3/2)/tref, 1/sec)
# Conversion of X to B
r1 = Reaction("r1", ["X"], ["B"],
              param_map={"kon": "kon", "koff": "koff"},
              eqn_f_str="X*kon - B*koff")

Create parameter and reaction containers and add in associated objects.

In [ ]:
pc = ParameterContainer()
pc.add([kon, koff])
rc = ReactionContainer()
rc.add([r1])

## Create/load in mesh

In SMART we have different levels of meshes. Here we create a UnitSquare mesh defined by

$$
\Omega = [0, 1] \times [0, 1] \subset \mathbb{R}^2
$$

which will serve as our parent mesh

For our two domains, we have two associated "child meshes", which are set by the marker functions `mf2` and `mf1`:
- surf: in this case, all cells (triangles) belong to this mesh; here, marked by `mf2 = 1`
- edge: 1D child mesh including all line elements along the edges of the domain; here, marked by `mf1 = 3`

Note that the marker values must be chosen to match those given in the compartment definitions above.

In [ ]:
useSpheroid = True
if useSpheroid:
    # rOuter = [0.6849, 0.4365, 2.1896]
    # rInner = [0.0,0.0,0.0]
    # domain, facet_markers, cell_markers = mesh_tools.create_ellipsoids(rOuter, rInner, hEdge=0.05)
    vol = Compartment("vol", 3, um, 1) # SMART just needs to know its a 3d mesh
    cc.add(vol)
    mesh_file = pathlib.Path("spheroid_ellipsoid_mesh.h5")
    # mesh_file = pathlib.Path("spheroid_ellipsoid_mesh_new.h5")
    # mesh_tools.write_mesh(domain, facet_markers, cell_markers, filename=mesh_file)
else:
    # define dimensions of domain
    Shat = 200
    x_size = np.sqrt(Shat/B.umax)
    y_size = np.sqrt(Shat/B.umax)
    # Create mesh
    m = 30
    n = int(x_size/y_size)*m
    rect_mesh = d.RectangleMesh(d.Point(0.0, 0.0), d.Point(x_size, y_size), n, m)
    mf2 = d.MeshFunction("size_t", rect_mesh, 2, 10)
    mf1 = d.MeshFunction("size_t", rect_mesh, 1, 0)
    class OuterEdge(d.SubDomain):
        def inside(self, x, on_boundary):
            return on_boundary
    outerEdge = OuterEdge()
    outerEdge.mark(mf1, 3)
    mesh_folder = pathlib.Path("rect_mesh")
    mesh_folder.mkdir(exist_ok=True)
    mesh_file = mesh_folder / "rect_mesh.h5"
    mesh_tools.write_mesh(rect_mesh, mf1, mf2, mesh_file)

Finally, we initialize the `mesh.ParentMesh` object, using the hdf5 file as input.

In [ ]:
parent_mesh = mesh.ParentMesh(
    mesh_filename=str(mesh_file),
    mesh_filetype="hdf5",
    name="parent_mesh",
)

## Initialize model and solver
Now we are ready to set up the model. First we load the default configurations and set the solver config.

In [ ]:
config_cur = config.Config()
config_cur.flags.update({"allow_unused_components": True})
config_cur.solver.update(
    {
        "final_t": 100.0,
        "initial_dt": 0.001,
        "time_precision": 8,
        "attempt_timestep_restart_on_divergence": True,
    }
)

We create the model object initialize the model using the `initialize` function found in the `smart.model` module. We then save the model information to a .pkl file for later reference.

Note that we could later load the model information from the pickle file using the line:
```
model_cur = model.from_pickle(model_cur.pkl)
```

In [ ]:
model_cur = model.Model(pc, sc, cc, rc, config_cur, parent_mesh)
model_cur.initialize()
model_cur.to_pickle('model_cur.pkl')

We then perturb the initial conditions by adding white noise to the dolfin vectors associated with each species.

In [ ]:
# add white noise perturbation to initial conditions
for sp_str in ("B"):#("X", "B"):
    sp = model_cur.sc[sp_str]
    u = model_cur.cc[sp.compartment_name].u["u"]
    indices = sp.dof_map
    uvec = u.vector()
    values = uvec.get_local()
    cur_seed = ord(sp_str)  # set seed for reproducibility
    generator_cur = np.random.default_rng(cur_seed)
    values[indices] = np.multiply(values[indices],
                                  generator_cur.normal(1, 0.01, len(indices)))
    uvec.set_local(values)
    uvec.apply("insert")
    nvec = model_cur.cc[sp.compartment_name].u["n"].vector()
    nvec.set_local(values)
    nvec.apply("insert")

Define other functions to be used for mass conservation and cutoffs here

In [ ]:
def _set_clipped_sum_c_tmp_scalar(sp, xi1_scalar, dt_val, epsilon):
    """clip(c + dt*xi_1) on all DOFs; xi_1 is a global scalar."""
    lo = sp.umax*(epsilon)
    hi = sp.umax*(1.0 - epsilon)
    c_tmp = d.Function(sp.V)
    d.assign(c_tmp, sp.sol)
    cvec = c_tmp.vector()[:]
    cvec = np.clip(cvec + float(dt_val) * xi1_scalar, float(lo), float(hi))
    c_tmp.vector().set_local(cvec)
    c_tmp.vector().apply("insert")
    return c_tmp

Xfunc = model_cur.sc["X"].sol
Xdof = model_cur.sc["X"].dof_map
Bfunc = model_cur.sc["B"].sol
Bdof = model_cur.sc["B"].dof_map
dx = d.Measure("dx", model_cur.cc["surf"].dolfin_mesh)
c_mass_init = d.assemble_mixed((Xfunc+Bfunc)*dx)
def F_mass(xi_arg1, epsilon, c_mass_init):
    """F(xi_1) = int clip(phi_X + dt*xi_1) + int clip(phi_B + dt*xi_1) minus previous X+B."""
    dt_val = float(model_cur.dt)
    Xtemp = _set_clipped_sum_c_tmp_scalar(
        model_cur.sc["X"], xi_arg1, dt_val, epsilon)
    Btemp = _set_clipped_sum_c_tmp_scalar(
        model_cur.sc["B"], xi_arg1, dt_val, epsilon)
    dx = d.Measure("dx", Xtemp.function_space().mesh())
    mass_err = d.assemble_mixed((Xtemp+Btemp)*dx) - c_mass_init
    return mass_err

def assign_from_sub(subfunc, func, dofmap):
    fullvec = func.vector()[:]
    subvec = subfunc.vector()[:]
    fullvec[dofmap] = subvec
    func.vector().set_local(fullvec)
    func.vector().apply("insert")

# check that the above functions are consistent
if F_mass(0.0, 0.01, c_mass_init) > 0.0:
    raise ValueError("This should not be possible")

## Solve the system and write output data
Now, we are ready to start the solution process. We store the initial conditions to output files and then solve the system at each time step using the `monolithic_solve` function. Once we pass the final time chosen above, we exit the loop.

In [ ]:
# Write initial condition(s) to file
results = dict()
result_folder = pathlib.Path("resultsRect")
result_folder.mkdir(exist_ok=True)
for species_name, species in model_cur.sc.items:
    results[species_name] = d.XDMFFile(
        model_cur.mpi_comm_world, str(result_folder / f"{species_name}.xdmf")
    )
    results[species_name].parameters["flush_output"] = True
    results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)

# Set loglevel to warning in order not to pollute notebook output
logger.setLevel(logging.WARNING)

epsilon = 0.001
xi_secant_max_iter = 500

# Solve
while True:
    print(f"Time is {model_cur.t}")
    # Solve the system
    model_cur.monolithic_solve()
    model_cur.adjust_dt()
    # Secant on global scalar xi_1: initial guesses (0, -dt).
    xi_secant_iter = 0
    xi_guess_prev = 0.0
    xi_guess = -float(model_cur.dt)
    secant_tol = 1e-12
    F1 = F_mass(xi_guess, epsilon, c_mass_init)
    F0 = F_mass(xi_guess_prev, epsilon, c_mass_init)
    while (xi_secant_iter < xi_secant_max_iter and 
           abs(F1) > secant_tol and abs(F0) > secant_tol):
        xi_secant_iter += 1
        denom = F1 - F0
        if abs(denom) < 1e-30 or xi_guess == xi_guess_prev:
            break
        xi_guess_next = xi_guess - F1 * (xi_guess - xi_guess_prev) / denom
        xi_guess_prev = float(xi_guess)
        xi_guess = float(xi_guess_next)
        F1 = F_mass(xi_guess, epsilon, c_mass_init)
        F0 = F_mass(xi_guess_prev, epsilon, c_mass_init)
    print(f"Secant approach converged in {xi_secant_iter} iterations")
    # now assign corrected values
    Xnew = _set_clipped_sum_c_tmp_scalar(model_cur.sc["X"], xi_guess, model_cur.dt, epsilon)
    assign_from_sub(Xnew, Xfunc, Xdof)
    Bnew = _set_clipped_sum_c_tmp_scalar(model_cur.sc["B"], xi_guess, model_cur.dt, epsilon)
    assign_from_sub(Bnew, Bfunc, Bdof)


    for species_name, species in model_cur.sc.items:
        results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)
    # End if we've passed the final time
    if model_cur.t >= model_cur.final_t:
        break